In [33]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "3"
!nvidia-smi
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import  sklearn
from sklearn.model_selection import train_test_split
#import ray
import os
import math
import matplotlib.pyplot as plt
import tensorflow.keras.optimizers.schedules as schedules
import pickle 

Thu Nov 16 14:27:18 2023       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 520.61.05    Driver Version: 520.61.05    CUDA Version: 11.8     |
|-------------------------------+----------------------+----------------------+
| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  NVIDIA RTX A6000    On   | 00000000:1A:00.0 Off |                  Off |
| 30%   36C    P2    68W / 300W |  45083MiB / 49140MiB |      1%      Default |
|                               |                      |                  N/A |
+-------------------------------+----------------------+----------------------+
|   1  NVIDIA RTX A6000    On   | 00000000:1B:00.0 Off |                  Off |
| 30%   

In [34]:
os.cpu_count()

32

In [35]:
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)


In [36]:
#ray.shutdown()
#ray.init(num_cpus=2, num_gpus=0)
#ray.shutdown()

In [37]:
originaldir = os.getcwd()

# Data processing

For the homework, you should download the dataset with the provided link. The provided dataset has already partitioned the data by clients.

In [38]:
os.chdir('/nfs/home/dem1110/Assignment 3/Part 1/P1/')
train_data = 'train_data.npy'
train_data_arr = np.load(train_data, allow_pickle=True)
test_data = 'test_data.npy'
test_data_arr = np.load(test_data, allow_pickle=True)


In [39]:
os.chdir(originaldir)
os.getcwd()

'/nfs/home/dem1110/Assignment 3/Part 1/P1/C=10, E=50, T=1000, B=16'

# Define model

In [40]:
## regular model 
'''
def get_model():
    model = tf.keras.models.Sequential(
        [ tf.keras.layers.Flatten(input_shape=(28, 28)),
            tf.keras.layers.Dense(128, activation="relu"),
            tf.keras.layers.Dense(128, activation="relu"),
            tf.keras.layers.Dense(62, activation="softmax")])
    model.compile("adam", "sparse_categorical_crossentropy", metrics=["accuracy"])
    return model
'''
## adding leanring rate.
def get_model(lr):
    lr_schedule = schedules.ExponentialDecay(
        initial_learning_rate=lr,
        decay_steps=10000,
        decay_rate=0.9,
        staircase=True
    )

    model = tf.keras.models.Sequential([
        tf.keras.layers.Flatten(input_shape=(28, 28)),
        tf.keras.layers.Dense(128, activation="relu"),
        tf.keras.layers.Dense(62, activation="softmax")
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr_schedule),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model


'''
## normalized data 
def get_model():
    model = tf.keras.models.Sequential(
        [ tf.keras.layers.Lambda(lambda x: x / 255.0, input_shape=(28, 28)),
            tf.keras.layers.Flatten(),
            tf.keras.layers.Dense(128, activation="relu"),
            tf.keras.layers.Dense(128, activation="relu"),
            tf.keras.layers.Dense(62, activation="softmax")])
    model.compile("adam", "sparse_categorical_crossentropy", metrics=[tf.keras.metrics.SparseCategoricalAccuracy()])
    return model
'''

'\n## normalized data \ndef get_model():\n    model = tf.keras.models.Sequential(\n        [ tf.keras.layers.Lambda(lambda x: x / 255.0, input_shape=(28, 28)),\n            tf.keras.layers.Flatten(),\n            tf.keras.layers.Dense(128, activation="relu"),\n            tf.keras.layers.Dense(128, activation="relu"),\n            tf.keras.layers.Dense(62, activation="softmax")])\n    model.compile("adam", "sparse_categorical_crossentropy", metrics=[tf.keras.metrics.SparseCategoricalAccuracy()])\n    return model\n'

## Splitting Data

In [41]:
all_train = []
all_val = []
x_train_all = []
y_train_all = []
x_val_all =[]
y_val_all =[]

for i in range(100):
    
    data = train_data_arr[i]
    # Split the data into training and validation sets
    images_train, images_val, labels_train, labels_val = train_test_split(
        data['images'],
        data['labels'],
        test_size=0.2,
        random_state=42)
    # Normalize the input data
    #images_train = np.array(images_train)
    #images_val = np.array(images_val)
    #images_train, images_val = images_train / 255.0, images_val / 255.0
    
    
    # Create TensorFlow datasets
    #expand dimensions of input to (1, 28, 28):
    #print("images before expansion: ", images_train.shape)
    #images_train = np.array([np.expand_dims(image, axis=0) for image in images_train])
    #images_val = [np.expand_dims(image, axis=0) for image in images_val]
    #images_train = np.expand_dims(images_train, axis=1)  # Add batch dimension
    #images_val = np.expand_dims(images_val, axis=1)
    #print("images expanded: ", images_train[0].shape)
    train_dataset = tf.data.Dataset.from_tensor_slices((images_train, labels_train))
    val_dataset = tf.data.Dataset.from_tensor_slices((images_val, labels_val))
    all_train.append(train_dataset)
    all_val.append(val_dataset)


# Client class

In [42]:
#@ray.remote(num_cpus=1, num_gpus=0)
class Client:
    def __init__(self, i, batch_size, initial_learning_rate, E):
        # Create model
        self.model = get_model(initial_learning_rate)
        self.training_data = all_train[i]
        self.validation_data = all_val[i]
        self.batch_size = batch_size
        self.E = E

    def train(self, global_weights):
        self.model.set_weights(global_weights)
        
        
        # Define a custom callback for printing validation metrics every 10 epochs
        '''
        class PrintValidationMetrics(tf.keras.callbacks.Callback):
            def on_epoch_end(self, epoch, logs=None):
                if epoch % 10 == 0:
                    val_loss, val_accuracy = self.model.evaluate(
                        self.validation_data.batch(self.batch_size), verbose=0)
                    print(f'Epoch {epoch}, Validation Loss: {val_loss:.4f}, Validation Accuracy: {val_accuracy:.4f}')

        callback = PrintValidationMetrics()
        
        '''
        
        self.model.fit(self.training_data.batch(self.batch_size), epochs=self.E, validation_data=self.validation_data.batch(self.batch_size), verbose=0)
        ### to get accuracy and loss as arrays for all epochs: 
        '''
         history = self.model.fit(self.training_data.batch(self.batch_size), epochs=E, validation_data=self.validation_data.batch(self.batch_size), verbose=0)
        
        # Access the training history
        ## define objects of these arrays in constructor
        train_loss = history.history['loss']
        train_accuracy = history.history['accuracy']
        val_loss = history.history['val_loss']
        val_accuracy = history.history['val_accuracy']

        return train_loss, train_accuracy, val_loss, val_accuracy

        '''
        return self.model.get_weights()

    def evaluate(self):
        #gives scalars of loss and accuracy at the end of training 
        train_loss, train_accuracy = self.model.evaluate(self.training_data.batch(self.batch_size), verbose=0)
        val_loss, val_accuracy = self.model.evaluate(self.validation_data.batch(self.batch_size))
        return train_loss, train_accuracy, val_loss, val_accuracy


# Aggregated metrics and losses
Aggregate client metrics and losses for (federated) evaluation metrics.

In [43]:
sample_nbs = []
for i in range(100):
    #print(i)
    #print(len(train_data_arr[i]['labels']))
    #print("train data: ", len( all_train[i]))
    #print("80% of local data: ", len(train_data_arr[i]['labels'])*0.8)
    sample_nbs.append(len(train_data_arr[i]['labels']))
#sample_nbs

In [44]:
def weighted_average(sampled_clients, metric):
    # TODO: Aggregate client losses or accuracies by taking a weighted average.
    #       Weights are proportional to the number of samples at each client
    sample_size_selected = [sample_nbs[i] for i in sampled_clients]
    #sorted list so order should match
    agg_train_acc = sum(weight * loss for weight, loss in zip(sample_size_selected, metric['train_accuracy'])) / sum(sample_size_selected)
    agg_train_loss = sum(weight * loss for weight, loss in zip(sample_size_selected, metric['train_loss'])) / sum(sample_size_selected)
    agg_val_acc = sum(weight * loss for weight, loss in zip(sample_size_selected, metric['val_accuracy'])) / sum(sample_size_selected)
    agg_val_loss = sum(weight * loss for weight, loss in zip(sample_size_selected, metric['val_loss'])) / sum(sample_size_selected)


    return agg_train_acc, agg_train_loss, agg_val_acc, agg_val_loss

# Aggregation of client updates

In [45]:
def fedAvg(sampled_clients, client_weights):
    # Get the size of samples for each client
    sample_size_selected = [sample_nbs[c] for c in sampled_clients]
    #print("shape of local weights: ", len(client_weights[0]), client_weights[0][0].shape)
    list_dim = len(client_weights[0])
    np_dim = client_weights[0][0].shape
    total = sum(sample_size_selected)
    
    avg_weights = []
    for k in range(list_dim):
        avg_weights.append(np.zeros(client_weights[0][k].shape))

    for i in range (len(sampled_clients)):
        scalar = sample_size_selected[i]/total
        for j in range(0, list_dim):
            this_numpy = np.multiply(client_weights[i][j], scalar)
            avg_weights[j] = np.add(avg_weights[j],this_numpy)  
    #print("shape of avg weights: ", len(avg_weights), avg_weights[0].shape)
    return avg_weights


In [46]:
def plots_train(normalized, t, client, E, batch_size,  train_loss, train_accuracy, val_loss, val_accuracy):
    plt.figure(figsize=(12, 4))
    #plt.subplot(1, 2, 1)
    plt.plot(train_loss, label='Training Loss')
    plt.plot(val_loss, label='Validation Loss')
    plt.title('Training and Validation Loss')
    plt.xlabel('Rounds')
    plt.ylabel('Loss')
    plt.legend()
    plt.savefig(normalized + '_train_loss_'+str(t)+'_'+str(client)+'_'+str(E)+'_'+str(batch_size)+'_.png')
    plt.close()
    # Plot training accuracy and validation accuracy
    plt.figure(figsize=(12, 4))
    plt.plot(train_accuracy, label='Training Accuracy')
    plt.plot(val_accuracy, label='Validation Accuracy')
    plt.title('Training and Validation Accuracy')
    plt.xlabel('Rounds')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.savefig(normalized + '_train_accuracy_'+str(t)+'_'+str(client)+'_'+str(E)+'_'+str(batch_size)+'_.png')
    plt.close()

# Launch the learning

In [ ]:
run = "p1-C=10, E=100"
Agg_metric = {}
Agg_metric['train_loss'] = []
Agg_metric['train_accuracy'] =[]
Agg_metric['val_loss'] =[]
Agg_metric['val_accuracy'] =[]
    
# Initialize the global model weights
batch_size = 16
# Initialize the global model weights with an initial learning rate
initial_learning_rate = 0.0001
global_model = get_model(initial_learning_rate)

#global_model = get_model()
users = np.arange(100)
# Get data
#data_by_clients, test_data = partition_data()

total_rounds = 1000

C = 10   # Only a fraction of C of clients participate in the local training per round
E = 50    # Number of epochs for each local training

for round in range(total_rounds):
    # Step 1
    print("communication round: ", round)
    #initiate empty list of model weights 
    client_weights =[]
    metric = {}
    metric['train_loss'] = []
    metric['train_accuracy'] =[]
    metric['val_loss'] =[]
    metric['val_accuracy'] =[] 
    # TODO: Sample C fraction of clients
    sampled_clients = np.random.choice(users, C, replace=False)
    sampled_clients = sorted(sampled_clients)
    # TODO: Get the local data for the selected clients
    for i in sampled_clients:
        print("client: ", i)
        #perform local training
        #define instance of client:
        current_client = Client(i, batch_size, initial_learning_rate, E) 
        
        #save local weights
        
        local_weights = current_client.train(global_model.get_weights())
        #print(local_weights[0].shape)
        client_weights.append(local_weights)
        #save local results
        
        train_loss, train_accuracy, val_loss, val_accuracy = current_client.evaluate()
        metric['train_loss'].append(train_loss)
        metric['train_accuracy'].append(train_accuracy)
        metric['val_loss'].append(val_loss)
        metric['val_accuracy'].append(val_accuracy)
        #print("train_loss : ", train_loss)
        #print("train_accuracy: ", train_accuracy)
        

    # Step 3
    # TODO: Aggregate the losses and metrics, keep track of the metrics
    agg_train_acc, agg_train_loss, agg_val_acc, agg_val_loss = weighted_average(sampled_clients, metric)
    Agg_metric['train_loss'].append(agg_train_loss)
    Agg_metric['train_accuracy'].append(agg_train_acc)
    Agg_metric['val_loss'].append(agg_val_loss)
    Agg_metric['val_accuracy'].append(agg_val_acc)
    print("aggregated validation accuracy: ",Agg_metric['val_accuracy'][round]) 
    # Step 4
    # TODO: Aggregate the client updates
    weighted_avg_weights = fedAvg(sampled_clients, client_weights)
    
    # Step 5
    # Update the global model
    global_model.set_weights(weighted_avg_weights)
    
    if round%50==0 and round!=0:


    # Specify the file path
        file_path = 'Agg_metric_run3_'+str(round)+'_'+str(E)+'_'+str(C)+'_'+str(batch_size)+'_.pickle'

        # Save the dictionary to a pickle file
        with open(file_path, 'wb') as file:
            pickle.dump(Agg_metric, file)

        print(f'Dictionary saved to {file_path}')

        global_model.save('global_model_'+str(round)+'_'+str(E)+'_'+str(C)+'_'+str(batch_size)+'_.h5')
        plots_train(run, round, C, E, batch_size,  Agg_metric['train_loss'] , Agg_metric['train_accuracy'], Agg_metric['val_loss'], Agg_metric['val_accuracy'])
        

communication round:  0
client:  12
4/4 [==============================] - 0s 4ms/step - loss: 4.0147 - accuracy: 0.2115
client:  15
5/5 [==============================] - 0s 3ms/step - loss: 3.6820 - accuracy: 0.1324
client:  41
4/4 [==============================] - 0s 3ms/step - loss: 3.6801 - accuracy: 0.1754
client:  48
4/4 [==============================] - 0s 2ms/step - loss: 3.5251 - accuracy: 0.2542
client:  58
5/5 [==============================] - 0s 2ms/step - loss: 3.6043 - accuracy: 0.1739
client:  70
5/5 [==============================] - 0s 3ms/step - loss: 3.6159 - accuracy: 0.2239
client:  74
4/4 [==============================] - 0s 3ms/step - loss: 3.3578 - accuracy: 0.3500
client:  91
5/5 [==============================] - 0s 2ms/step - loss: 3.5053 - accuracy: 0.2361
client:  93
4/4 [==============================] - 0s 3ms/step - loss: 4.1897 - accuracy: 0.1579
client:  99
5/5 [==============================] - 0s 2ms/step - loss: 3.3492 - accuracy: 0.2571
aggreg

2023-11-16 18:33:34.469570: W tensorflow/core/data/root_dataset.cc:266] Optimization loop failed: CANCELLED: Operation was cancelled


5/5 [==============================] - 0s 2ms/step - loss: 0.7864 - accuracy: 0.8421
client:  14
4/4 [==============================] - 0s 3ms/step - loss: 2.3112 - accuracy: 0.6400
client:  21
5/5 [==============================] - 0s 2ms/step - loss: 0.8681 - accuracy: 0.8088
client:  32
5/5 [==============================] - 0s 2ms/step - loss: 2.1602 - accuracy: 0.7606
client:  53
5/5 [==============================] - 0s 2ms/step - loss: 1.4000 - accuracy: 0.7568
client:  57
5/5 [==============================] - 0s 2ms/step - loss: 0.6344 - accuracy: 0.8533
client:  65
5/5 [==============================] - 0s 2ms/step - loss: 1.0125 - accuracy: 0.8289
client:  74
4/4 [==============================] - 0s 2ms/step - loss: 1.7381 - accuracy: 0.7333
client:  90
5/5 [==============================] - 0s 2ms/step - loss: 1.2026 - accuracy: 0.7465
aggregated validation accuracy:  0.7723022136578969
communication round:  449
client:  5
5/5 [==============================] - 0s 3ms/step

2023-11-16 18:35:57.962187: W tensorflow/core/data/root_dataset.cc:266] Optimization loop failed: CANCELLED: Operation was cancelled


4/4 [==============================] - 0s 2ms/step - loss: 1.1474 - accuracy: 0.7143
client:  95
5/5 [==============================] - 0s 2ms/step - loss: 1.5833 - accuracy: 0.6970
aggregated validation accuracy:  0.7103752748297137
communication round:  452
client:  0
5/5 [==============================] - 0s 3ms/step - loss: 1.3203 - accuracy: 0.7536
client:  2
4/4 [==============================] - 0s 2ms/step - loss: 2.0666 - accuracy: 0.6719
client:  15
5/5 [==============================] - 0s 2ms/step - loss: 1.3933 - accuracy: 0.6471
client:  16
5/5 [==============================] - 0s 10ms/step - loss: 0.8910 - accuracy: 0.8133
client:  47
5/5 [==============================] - 0s 2ms/step - loss: 1.4618 - accuracy: 0.8358
client:  48
4/4 [==============================] - 0s 2ms/step - loss: 1.9264 - accuracy: 0.6780
client:  57
5/5 [==============================] - 0s 2ms/step - loss: 0.5672 - accuracy: 0.8533
client:  64
5/5 [==============================] - 0s 2ms/step

...

client:  81
3/3 [==============================] - 0s 3ms/step - loss: 5.2521 - accuracy: 0.4545
client:  84
4/4 [==============================] - 0s 2ms/step - loss: 2.3117 - accuracy: 0.6250
client:  94
5/5 [==============================] - 0s 2ms/step - loss: 1.3401 - accuracy: 0.8718
aggregated validation accuracy:  0.7200877874516144
communication round:  766
client:  2
4/4 [==============================] - 0s 2ms/step - loss: 2.8183 - accuracy: 0.6875
client:  10
5/5 [==============================] - 0s 2ms/step - loss: 1.6269 - accuracy: 0.7895
client:  17
5/5 [==============================] - 0s 3ms/step - loss: 0.9380 - accuracy: 0.8429
client:  21
5/5 [==============================] - 0s 3ms/step - loss: 1.0362 - accuracy: 0.7941
client:  22
5/5 [==============================] - 0s 2ms/step - loss: 1.5497 - accuracy: 0.7231
client:  25
2/2 [==============================] - 0s 5ms/step - loss: 7.9396 - accuracy: 0.5417
client:  43
4/4 [==============================] -

2023-11-17 00:48:22.714869: W tensorflow/core/data/root_dataset.cc:266] Optimization loop failed: CANCELLED: Operation was cancelled


4/4 [==============================] - 0s 9ms/step - loss: 0.3382 - accuracy: 0.9107
client:  97
5/5 [==============================] - 0s 16ms/step - loss: 2.5549 - accuracy: 0.6308
client:  98
3/3 [==============================] - 0s 11ms/step - loss: 1.1831 - accuracy: 0.7317
aggregated validation accuracy:  0.7320582719040486
communication round:  900
client:  0
5/5 [==============================] - 0s 12ms/step - loss: 1.3938 - accuracy: 0.7536
client:  1
5/5 [==============================] - 0s 5ms/step - loss: 0.9662 - accuracy: 0.7733
client:  10
5/5 [==============================] - 0s 16ms/step - loss: 1.6466 - accuracy: 0.8026
client:  13
5/5 [==============================] - 0s 7ms/step - loss: 1.4650 - accuracy: 0.7297
client:  32
5/5 [==============================] - 0s 5ms/step - loss: 2.7180 - accuracy: 0.7324
client:  33
4/4 [==============================] - 0s 25ms/step - loss: 2.6751 - accuracy: 0.6562
client:  46
3/3 [==============================] - 0s 6ms/

2023-11-17 01:43:04.462526: W tensorflow/core/data/root_dataset.cc:266] Optimization loop failed: CANCELLED: Operation was cancelled


4/4 [==============================] - 0s 13ms/step - loss: 2.3844 - accuracy: 0.6724
client:  54
5/5 [==============================] - 0s 15ms/step - loss: 1.7203 - accuracy: 0.7778
client:  58
5/5 [==============================] - 0s 7ms/step - loss: 0.7987 - accuracy: 0.7826
client:  66
4/4 [==============================] - 0s 12ms/step - loss: 3.4177 - accuracy: 0.6429
client:  78
5/5 [==============================] - 0s 9ms/step - loss: 2.1809 - accuracy: 0.6866
client:  87
4/4 [==============================] - 0s 7ms/step - loss: 1.7269 - accuracy: 0.8421
aggregated validation accuracy:  0.7190918197200248
communication round:  920
client:  5
5/5 [==============================] - 0s 5ms/step - loss: 1.1663 - accuracy: 0.7662
client:  18
5/5 [==============================] - 0s 15ms/step - loss: 2.5296 - accuracy: 0.6849
client:  30
4/4 [==============================] - 0s 21ms/step - loss: 2.7219 - accuracy: 0.7460
client:  46
3/3 [==============================] - 0s 20m

# Create plots

In [ ]:
def test(test_data_arr, weights, batch_size, lr):
    model = get_model(lr)  # Construct the model
    model.set_weights(weights)  # Set model weights with the latest parameters

    # Assuming test_data is a tuple of test_images and test_labels
    test_images = np.array(test_data_arr[0]['images'])
    test_labels = np.array(test_data_arr[0]['labels'])
    
 

    # Normalize the input data
    #test_images = np.array(test_images)
    #test_images= test_images / 255.0
    
    test_images_flat = test_images.reshape(test_images.shape[0], -1)
    
    test_dataset = tf.data.Dataset.from_tensor_slices((test_images, test_labels))
    batch_size = batch_size  # Set your desired batch size
    # Evaluate the model using the dataset
    batched_test_dataset = test_dataset.batch(batch_size)

    test_loss, test_accuracy = model.evaluate(batched_test_dataset)


    print(f"Test Accuracy: {test_accuracy:.4f}")

    return test_loss, test_accuracy

In [ ]:
plots_train(run, total_rounds, C, E, batch_size, Agg_metric['train_loss'] , Agg_metric['train_accuracy'], Agg_metric['val_loss'], Agg_metric['val_accuracy'])
# Evaluate on test data after training
test_loss, test_accuracy = test(test_data_arr, global_model.get_weights(), batch_size, initial_learning_rate)

In [ ]:
print("test loss: ", test_loss)
print("test accuracy: ", test_accuracy)

In [ ]:
print("last train acc: ", Agg_metric['train_accuracy'][999])
print("last validation acc: ", Agg_metric['val_accuracy'][999])

In [ ]:
test_results = np.array ([test_loss, test_accuracy])
np.save('test_results_'+run+'_'+str(total_rounds)+'_'+str(C)+'_'+str(E)+'_'+str(batch_size)+'_.npy', test_results)